In [17]:
# A. Menyiapkan Streaming Source
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Mempersiapkan skema eksplisit untuk data pesanan Kopi Nusantara
skema_tugas = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Menyiapkan readStream dari folder simulasi tugas
df_tugas = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas)

print("SparkSession siap. Folder tugas10_stream dan readStream berhasil disiapkan.")

SparkSession siap. Folder tugas10_stream dan readStream berhasil disiapkan.


In [19]:
# Hentikan semua query yang masih aktif di background
for q in spark.streams.active:
    q.stop()

print("Semua query aktif berhasil dihentikan!")

Semua query aktif berhasil dihentikan!


In [20]:
from pyspark.sql.functions import col, sum as spark_sum, count

# Hentikan query aktif jika ada
for q in spark.streams.active:
    if q.name == "agregasi_cabang":
        q.stop()

# Menambahkan kolom turunan total_harga (qty x harga_satuan)
df_tugas_with_total = df_tugas.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Menghitung akumulasi total pendapatan dan jumlah pesanan per cabang
agregasi_cabang = df_tugas_with_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query dengan sink memori
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Streaming query 'agregasi_cabang' berhasil dimulai.")

Streaming query 'agregasi_cabang' berhasil dimulai.


In [21]:
from pyspark.sql.functions import col, window, sum as spark_sum, count

# Hentikan query aktif jika ada
for q in spark.streams.active:
    if q.name == "agregasi_waktu":
        q.stop()

# Membaca ulang stream dari folder yang sama untuk query kedua
df_tugas_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_tugas) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi total pendapatan berdasarkan window waktu 2 menit
agregasi_waktu = df_tugas_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query window waktu dengan sink memori
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Streaming query 'agregasi_waktu' berhasil dimulai.")

Streaming query 'agregasi_waktu' berhasil dimulai.


In [24]:
import time
import pandas as pd
from datetime import datetime

waktu_sekarang = datetime.now()

# Batch 1
pd.DataFrame({
    "pesanan_id": ["P1", "P2", "P3"],
    "nama_cabang": ["Magelang", "Solo", "Magelang"],
    "nama_menu": ["Kopi Kenangan", "Es Teh", "Kopi Tubruk"],
    "qty": [2, 1, 3],
    "harga_satuan": [18000, 5000, 12000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 3
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("Batch 1 terkirim. Menunggu pemrosesan...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 2
pd.DataFrame({
    "pesanan_id": ["P4", "P5"],
    "nama_cabang": ["Yogyakarta", "Solo"],
    "nama_menu": ["Kopi Susu", "Matcha Latte"],
    "qty": [1, 2],
    "harga_satuan": [20000, 25000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("Batch 2 terkirim. Menunggu pemrosesan...")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang").show()

# Batch 3
pd.DataFrame({
    "pesanan_id": ["P6", "P7"],
    "nama_cabang": ["Magelang", "Yogyakarta"],
    "nama_menu": ["Americano", "Kopi Susu"],
    "qty": [2, 2],
    "harga_satuan": [15000, 20000],
    "waktu_pesan": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("Batch 3 terkirim. Menunggu pemrosesan...")
time.sleep(6)

print("--- Hasil Akhir Agregasi Per Cabang ---")
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

print("--- Hasil Akhir Agregasi Window Waktu ---")
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu").show(truncate=False)

Batch 1 terkirim. Menunggu pemrosesan...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           55000|             2|
| Yogyakarta|           60000|             2|
|   Magelang|          102000|             3|
+-----------+----------------+--------------+

Batch 2 terkirim. Menunggu pemrosesan...
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|       Solo|           55000|             2|
| Yogyakarta|           60000|             2|
|   Magelang|          102000|             3|
+-----------+----------------+--------------+

Batch 3 terkirim. Menunggu pemrosesan...
--- Hasil Akhir Agregasi Per Cabang ---
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          102000|             3|
| Yogya

In [25]:
import os
import shutil
import time
import pandas as pd
from datetime import datetime

folder_output = "tugas10_output"
folder_checkpoint = "tugas10_checkpoint"

shutil.rmtree(folder_output, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)

# Menuliskan data mentah df_tugas secara kontinu ke format Parquet
query3 = df_tugas.writeStream \
    .format("parquet") \
    .option("path", folder_output) \
    .option("checkpointLocation", folder_checkpoint) \
    .outputMode("append") \
    .start()

# Mengirim batch ke-4 untuk menguji penulisan ke Parquet sink
pd.DataFrame({
    "pesanan_id": ["P8", "P9"],
    "nama_cabang": ["Magelang", "Semarang"],
    "nama_menu": ["V60", "Espresso"],
    "qty": [1, 2],
    "harga_satuan": [22000, 15000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")] * 2
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

time.sleep(6)

# Membuktikan berkas Parquet tersimpan dan dapat dibaca sebagai DataFrame batch biasa
df_parquet = spark.read.parquet(folder_output)
print("--- Data Mentah Terbaca Kembali dari File Parquet ---")
df_parquet.show(truncate=False)

# Menghentikan seluruh streaming query dan menutup sesi Spark
for q in spark.streams.active:
    q.stop()

spark.stop()
print("Seluruh streaming query dihentikan dan SparkSession ditutup secara aman.")

--- Data Mentah Terbaca Kembali dari File Parquet ---
+----------+-----------+-------------+---+------------+-------------------+
|pesanan_id|nama_cabang|nama_menu    |qty|harga_satuan|waktu_pesan        |
+----------+-----------+-------------+---+------------+-------------------+
|P1        |Magelang   |Kopi Kenangan|2  |18000       |2026-11-11 14:31:18|
|P2        |Solo       |Es Teh       |1  |5000        |2026-11-11 14:31:18|
|P3        |Magelang   |Kopi Tubruk  |3  |12000       |2026-11-11 14:31:18|
|P6        |Magelang   |Americano    |2  |15000       |2026-11-11 14:31:18|
|P7        |Yogyakarta |Kopi Susu    |2  |20000       |2026-11-11 14:31:18|
|P4        |Yogyakarta |Kopi Susu    |1  |20000       |2026-11-11 14:31:18|
|P5        |Solo       |Matcha Latte |2  |25000       |2026-11-11 14:31:18|
|P8        |Magelang   |V60          |1  |22000       |2026-11-11 14:33:28|
|P9        |Semarang   |Espresso     |2  |15000       |2026-11-11 14:33:28|
+----------+-----------+----------